# EDGAR Financial RAG System

## Overview

A natural language interface to SEC filings (10-K and 10-Q) for five hyperscalers, a regulated utility, an AI cloud provider and a launch/satellite company:
- **AMZN** - Amazon
- **GOOGL** - Alphabet/Google
- **META** - Meta/Facebook
- **MSFT** - Microsoft (fiscal year ends June)
- **ORCL** - Oracle (fiscal year ends May)
- **D** - Dominion Energy
- **CRWV** - CoreWeave (from IPO, March 2025)
- **SPCX** - SpaceX (from IPO, June 2026; includes the 424B4 prospectus, no 10-K yet)

**Data Coverage:** January 2020 - Present

## Architecture

All logic lives in the `edgar_rag` package; this notebook just calls it.

### 1. Structured Financial Data (XBRL)
- Facts from the balance sheet, income statement and cash flow statement
- **Every value carries its exact period** - discrete fiscal quarter, fiscal year, or balance-sheet date - so quarterly series are never mixed with year-to-date totals
- Quarters a filing doesn't report on its own (Q4, and cash-flow quarters, which 10-Qs only report year-to-date) are **derived** from year-to-date differences and flagged as such
- **Use for:** revenue, capex, expenses, assets, debt

### 2. Narrative Content (Vector RAG)
- Text chunks from risk factors, MD&A, business descriptions
- Voyage AI embeddings, searched with pgvector (HNSW index)
- **Use for:** risks, strategy, qualitative discussion

### 3. Query Router
- Claude extracts the query type, companies, line-item keywords, quarterly vs annual granularity and date window
- Retrieves from the relevant source(s), balanced per company, and synthesizes an answer

## Quick Start

```python
ask("How much did Amazon spend on purchases of property and equipment each quarter of 2025?")
ask("What risks does Microsoft mention about AI and power?")
ask("What does SpaceX say about Starlink capacity and capital spending?")
```

## Data Freshness

To pull in new filings (incremental - only filings not yet ingested), from the project root:

```bash
python -m edgar_rag update      # fetch + parse + embed new filings
python -m edgar_rag status      # per-company coverage
```

Then restart this notebook's kernel.


In [1]:
import pandas as pd
from edgar_rag import config
from edgar_rag.query import ask, query_edgar, search_financial_facts, search_narrative

pd.set_option("display.max_colwidth", 80)
print(f"✓ Companies: {', '.join(config.TICKERS)}")
print(f"✓ Claude model: {config.CLAUDE_MODEL}")
print(f"✓ Embedding model: {config.EMBEDDING_MODEL}")


✓ Companies: AMZN, GOOGL, META, MSFT, ORCL, D, CRWV, SPCX
✓ Claude model: claude-sonnet-5-5
✓ Embedding model: voyage-3-lite


/home/daniel/projects/edgar_rag/power/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Example Queries

In [2]:
# Example 1: Quantitative query
result = query_edgar(
    "How much did Amazon spend on technology and infrastructure in 2025?",
    synthesize=True
)

if 'financial_data' in result:
    print("\n" + "="*60)
    print("FINANCIAL DATA")
    print("="*60)
    display(result['financial_data'][['ticker', 'fiscal_year', 'fiscal_quarter', 'fiscal_period', 'period_start', 'period_end', 'label', 'value', 'unit', 'is_derived']])

if 'answer' in result:
    print("\n" + "="*60)
    print("SYNTHESIZED ANSWER")
    print("="*60)
    print(result['answer'])



QUERY: How much did Amazon spend on technology and infrastructure in 2025?



Query type: quantitative
Tickers: ['AMZN']
SQL keywords: ['Technology and infrastructure', 'Technology and content', 'Purchase of property and equipment']
Granularity: annual
Date range: 2025-01-01 to 2025-12-31

🔢 Searching financial facts...
   Found 2 financial facts

🤖 Synthesizing answer with Claude...



FINANCIAL DATA


,ticker,fiscal_year,fiscal_quarter,fiscal_period,period_start,period_end,label,value,unit,is_derived
0,AMZN,2025,None,FY,2025-01-01,2025-12-31,Purchases of property and equipment,131819.0,USD_millions,False
1,AMZN,2025,None,FY,2025-01-01,2025-12-31,Technology and infrastructure,108521.0,USD_millions,False



SYNTHESIZED ANSWER
Amazon reported **technology and infrastructure expense of $108,521 million** for fiscal year 2025 (January 1 – December 31, 2025), according to its 10-K filed 2026-02-06.

Note: this is the operating expense line item. It is separate from capital spending. In the same filing, Amazon's purchases of property and equipment were $131,819 million for FY2025. Because the question asks about spending on technology and infrastructure, I've used the expense line item, but the data doesn't break down how much of the capital expenditures relates to technology infrastructure.


In [3]:
# Example 2: Qualitative query
result = query_edgar(
    "What does Amazon mention related to data center power and energy?",
    synthesize=True
)

if 'narrative_data' in result:
    print("\n" + "="*60)
    print("NARRATIVE DATA")
    print("="*60)
    for idx, row in result['narrative_data'].head(3).iterrows():
        print(f"\n{idx+1}. {row['ticker']} {row['filing_type']} ({row['filing_date']})")
        print(f"   Section: {row['section'][:60]}...")
        print(f"   Similarity: {row['similarity']:.3f}")
        print(f"   {row['chunk_text'][:400]}...\n")

if 'answer' in result:
    print("\n" + "="*60)
    print("SYNTHESIZED ANSWER")
    print("="*60)
    print(result['answer'])


QUERY: What does Amazon mention related to data center power and energy?



Query type: qualitative
Tickers: ['AMZN']
SQL keywords: []
Granularity: any
Narrative query: Amazon AWS data center power energy availability electricity costs renewable energy supply constraints capacity

📄 Searching narrative chunks...
   Found 5 relevant chunks

🤖 Synthesizing answer with Claude...



NARRATIVE DATA

1. AMZN 10-Q (2024-08-02)
   Section: Results of Operations...
   Similarity: 0.521
   Technology and Infrastructure

Technology and infrastructure costs include payroll and related expenses for employees involved in the research and development of new and existing products and services, development, design, and maintenance of our stores, curation and display of products and services made available in our online stores, and infrastructure costs. Infrastructure costs include servers,...


2. AMZN 10-Q (2024-11-01)
   Section: Results of Operations...
   Similarity: 0.521
   Technology and Infrastructure

Technology and infrastructure costs include payroll and related expenses for employees involved in the research and development of new and existing products and services, development, design, and maintenance of our stores, curation and display of products and services made available in our online stores, and infrastructure costs. Infrastructure costs include servers,...

In [4]:
# Example 3: Comparative query (both quantitative and qualitative)
result = query_edgar(
    "Compare data center infrastructure spending across Amazon, Microsoft, and Google",
    max_financial_results=30,
    synthesize=True
)

if 'financial_data' in result:
    print("\n" + "="*60)
    print("FINANCIAL DATA")
    print("="*60)
    display(result['financial_data'][['ticker', 'fiscal_year', 'fiscal_quarter', 'fiscal_period', 'period_start', 'period_end', 'label', 'value', 'unit', 'is_derived']])

if 'answer' in result:
    print("\n" + "="*60)
    print("SYNTHESIZED ANSWER")
    print("="*60)
    print(result['answer'])



QUERY: Compare data center infrastructure spending across Amazon, Microsoft, and Google



Query type: both
Tickers: ['AMZN', 'MSFT', 'GOOGL']
SQL keywords: ['Purchase of property and equipment', 'Additions to property and equipment', 'Property and equipment, net', 'Depreciation and amortization']
Granularity: any
Narrative query: data center infrastructure capital expenditures investment in servers, data centers, and cloud capacity expansion

🔢 Searching financial facts...
   Found 90 financial facts

📄 Searching narrative chunks...


   Found 6 relevant chunks

🤖 Synthesizing answer with Claude...



FINANCIAL DATA


,ticker,fiscal_year,fiscal_quarter,fiscal_period,period_start,period_end,label,value,unit,is_derived
0,AMZN,2026,3.0,Q,2026-07-01,2026-09-30,Depreciation and amortization of property and equipment and capitalized cont...,20949.0,USD_millions,False
1,AMZN,2026,2.0,Q,2026-04-01,2026-06-30,Depreciation and amortization of property and equipment and capitalized cont...,19988.0,USD_millions,False
2,AMZN,2026,1.0,Q,2026-01-01,2026-03-31,Depreciation and amortization of property and equipment and capitalized cont...,18945.0,USD_millions,False
3,AMZN,2025,NaN,FY,2025-01-01,2025-12-31,Depreciation and amortization of property and equipment and capitalized cont...,65756.0,USD_millions,False
4,AMZN,2025,4.0,Q,2025-10-01,2025-12-31,Depreciation and amortization of property and equipment and capitalized cont...,19471.0,USD_millions,True
...,...,...,...,...,...,...,...,...,...,...
85,MSFT,2025,1.0,instant,None,2024-09-30,"Property and equipment, net of accumulated depreciation of $80,517 and $76,421",152863.0,USD_millions,False
86,MSFT,2025,2.0,instant,None,2024-12-31,"Property and equipment, net of accumulated depreciation of $82,820 and $76,421",166902.0,USD_millions,False
87,MSFT,2025,3.0,instant,None,2025-03-31,"Property and equipment, net of accumulated depreciation of $87,074 and $76,421",183939.0,USD_millions,False
88,MSFT,2024,4.0,instant,None,2024-06-30,"Property and equipment, net of accumulated depreciation of $93,653 and $76,421",135591.0,USD_millions,False



SYNTHESIZED ANSWER
## Capital spending comparison: Amazon, Microsoft and Google

None of the three companies reports a data-center-only figure in this data. The closest proxy is purchases of (or additions to) property and equipment (capex), plus net property and equipment (PP&E).

| Metric ($M) | AMZN | GOOGL | MSFT (FY ends June) |
|---|---|---|---|
| Capex, Jul–Sep 2026 | 77,259 (FY26 Q3) | 51,355 (FY26 Q3, derived) | 39,367 (FY27 Q1) |
| Capex, Jan–Sep 2026 (my sum of quarters) | 175,670 | 131,953 | n/a |
| Capex, CY2025 | 131,819 | 91,447 | FY2026 (Jul 2025–Jun 2026): 115,948 |
| Capex, CY2024 | 82,999 | 52,535 | not in data |
| Net PP&E, 9/30/2026 | 508,862 | 372,827 | 346,844 |
| Net PP&E, 9/30/2025 | 324,435 | 223,787 | 230,861 |
| Net PP&E growth, year over year (my calculation) | ~+57% | ~+67% | ~+50% |

Sources: AMZN 10-Q filed 2026-10-30 and 10-K filed 2026-02-06; GOOGL 10-Q filed 2026-10-29 and 10-K filed 2026-02-05; MSFT 10-Q filed 2026-10-28 and 10-K filed 2026-07-29.

*

### Quarterly capex (discrete quarters, Q4 derived from the 10-K)

In [5]:
ask("How much did Amazon spend on purchases of property and equipment each quarter of 2025?")


QUERY: How much did Amazon spend on purchases of property and equipment each quarter of 2025?



Query type: quantitative
Tickers: ['AMZN']
SQL keywords: ['Purchase of property and equipment', 'Purchases of property and equipment', 'property and equipment']
Granularity: quarterly
Date range: 2025-01-01 to 2025-12-31

🔢 Searching financial facts...
   Found 16 financial facts

🤖 Synthesizing answer with Claude...



ANSWER
Amazon's purchases of property and equipment in each quarter of 2025 were:

| Quarter | Period | Purchases of property and equipment | Source |
|---|---|---|---|
| FY2025 Q1 | Jan 1 – Mar 31, 2025 | $25,019M | 10-Q filed 2026-04-30 |
| FY2025 Q2 | Apr 1 – Jun 30, 2025 | $32,183M | 10-Q filed 2026-07-31 |
| FY2025 Q3 | Jul 1 – Sep 30, 2025 | $35,095M | 10-Q filed 2026-10-30 |
| FY2025 Q4 | Oct 1 – Dec 31, 2025 | $39,522M | 10-K filed 2026-02-06 (derived: full-year total minus year-to-date) |

The four quarters sum to $131,819M for 2025. This is my own addition, not a figure reported in the data.

Spending rose every quarter. Q4 was about 58% higher than Q1.

- **Q4 is derived.** The 10-K does not report it on its own.
- **Filing dates look odd.** Several of the cited filing dates (such as 10-Qs filed in 2026 for 2025 quarters) are what the data provides. Per the data notes, these are the latest reported figures for each period, so restated figures take precedence.
- **These are 

{'analysis': {'query_type': 'quantitative',
  'tickers': ['AMZN'],
  'sql_keywords': ['Purchase of property and equipment',
   'Purchases of property and equipment',
   'property and equipment'],
  'period_granularity': 'quarterly',
  'date_start': '2025-01-01',
  'date_end': '2025-12-31',
  'narrative_search_query': ''},
 'financial_data':    ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0    AMZN         2025               4             Q   2025-10-01  2025-12-31   
 1    AMZN         2025               3             Q   2025-07-01  2025-09-30   
 2    AMZN         2025               2             Q   2025-04-01  2025-06-30   
 3    AMZN         2025               1             Q   2025-01-01  2025-03-31   
 4    AMZN         2025               4             Q   2025-10-01  2025-12-31   
 5    AMZN         2025               3             Q   2025-07-01  2025-09-30   
 6    AMZN         2025               2             Q   2025-04-01  2025-06-30   
 7

### SpaceX (prospectus + 10-Qs)

In [6]:
ask("What does SpaceX say about Starlink and its capital expenditure plans?")


QUERY: What does SpaceX say about Starlink and its capital expenditure plans?



Query type: both
Tickers: ['SPCX']
SQL keywords: ['Purchase of property and equipment', 'Property and equipment, net', 'Depreciation and amortization']
Granularity: any
Narrative query: SpaceX Starlink satellite network capital expenditure plans, expected capex spending, investment in satellites, launch infrastructure and ground facilities

🔢 Searching financial facts...
   Found 15 financial facts

📄 Searching narrative chunks...
   Found 5 relevant chunks

🤖 Synthesizing answer with Claude...



ANSWER
# SpaceX (SPCX): Starlink and Capital Expenditure Plans

## What SpaceX says about Starlink (424B4 filed 2026-06-12)
- **Scale:** About 9,600 Starlink broadband and mobile satellites were in LEO as of March 31, 2026. SpaceX says it owns and operates about 75% of all active maneuverable satellites.
- **Next-generation V3 satellites:** SpaceX expects to begin deploying them on Starship in the second half of 2026. Each is designed for 1 Tbps of downlink capacity. One Starship launch could deploy up to 60 V3 satellites, a potential twenty-fold increase in Starlink downlink capacity deployed versus a Falcon 9 launch.
- **Enterprise:** Starlink's high-speed, low-latency internet is offered to customers in construction, agriculture, retail, telecom, hospitality, aviation, maritime and land mobility. Example sites include remote worksites, drilling rigs, rural hospitals, aircraft, cruise ships and trains.
- **AI link:** After the xAI acquisition, SpaceX says Starlink would provide low-

{'analysis': {'query_type': 'both',
  'tickers': ['SPCX'],
  'sql_keywords': ['Purchase of property and equipment',
   'Property and equipment, net',
   'Depreciation and amortization'],
  'period_granularity': 'any',
  'date_start': '',
  'date_end': '',
  'narrative_search_query': 'SpaceX Starlink satellite network capital expenditure plans, expected capex spending, investment in satellites, launch infrastructure and ground facilities'},
 'financial_data':    ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0    SPCX         2026               3             Q   2026-07-01  2026-09-30   
 1    SPCX         2025               3             Q   2025-07-01  2025-09-30   
 2    SPCX         2026               3       instant         None  2026-09-30   
 3    SPCX         2026               2       instant         None  2026-06-30   
 4    SPCX         2025               4       instant         None  2025-12-31   
 5    SPCX         2026               3       

## Interactive Query Interface

`ask(question)` prints the answer and returns the full result (analysis, retrieved facts and chunks).

In [7]:
ask("What is Microsoft's total stockholders equity trend over the last 2 years?")


QUERY: What is Microsoft's total stockholders equity trend over the last 2 years?



Query type: quantitative
Tickers: ['MSFT']
SQL keywords: ['Total stockholders equity', 'Total stockholders equity', 'Total liabilities and stockholders equity']
Granularity: annual
Date range: 2024-11-08 to 2026-11-08

🔢 Searching financial facts...
   Found 4 financial facts

🤖 Synthesizing answer with Claude...



ANSWER
**Microsoft's total stockholders' equity rose over the two fiscal year-ends in the data.**

| Fiscal year-end | Total stockholders' equity | Source |
|---|---|---|
| June 30, 2025 (FY2025 Q4) | $343,479M | 10-K filed 2026-07-29 |
| June 30, 2026 (FY2026 Q4) | $442,387M | 10-Q filed 2026-10-28 |

- **Change:** Equity increased by $98,908M, or about 28.8%, from June 30, 2025 to June 30, 2026.
- **Context:** Total liabilities and stockholders' equity (total assets) grew from $619,003M to $758,376M over the same period. Equity was about 55.5% of the total at June 30, 2025 and about 58.3% at June 30, 2026.

**Limits of the data:**
- Only two data points are provided, so I can't show an intermediate quarterly path or a June 30, 2024 starting point. The trend is therefore a single year-over-year comparison, not a full two-year series.
- The data doesn't explain what drove the increase, such as net income, stock issuance, or repurchases and dividends.



{'analysis': {'query_type': 'quantitative',
  'tickers': ['MSFT'],
  'sql_keywords': ['Total stockholders equity',
   'Total stockholders equity',
   'Total liabilities and stockholders equity'],
  'period_granularity': 'annual',
  'date_start': '2024-11-08',
  'date_end': '2026-11-08',
  'narrative_search_query': ''},
 'financial_data':   ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0   MSFT         2026               4       instant         None  2026-06-30   
 1   MSFT         2025               4       instant         None  2025-06-30   
 2   MSFT         2026               4       instant         None  2026-06-30   
 3   MSFT         2025               4       instant         None  2025-06-30   
 
   statement_type                                       label     value  \
 0  balance_sheet  Total liabilities and stockholders’ equity  758376.0   
 1  balance_sheet  Total liabilities and stockholders’ equity  619003.0   
 2  balance_sheet            

In [8]:
ask("What does Meta say about AI infrastructure investments?")


QUERY: What does Meta say about AI infrastructure investments?



Query type: qualitative
Tickers: ['META']
SQL keywords: ['Purchase of property and equipment', 'Property and equipment, net']
Granularity: any
Narrative query: Meta AI infrastructure investments capital expenditures data centers servers GPUs outlook strategy

📄 Searching narrative chunks...
   Found 5 relevant chunks

🤖 Synthesizing answer with Claude...



ANSWER
# Meta on AI infrastructure investments

**Direction of spending: increasing**
- **10-Q filed 2026-10-22 (Risk Factors):** Meta says it has "significantly increased our infrastructure investments in connection with our AI initiatives, and expect[s] our investments to continue to increase." It adds that "if our investments are not successful longer-term, our business and financial performance could be harmed."
- **10-Qs filed 2026-04-30 and 2026-07-30 (Material Cash Requirements):** Both use the same language: "We have increased investments in infrastructure and AI initiatives and expect to continue to do so."

**Funding**
- The same two 10-Qs say Meta expects available funds, cash flow from operations and financing activities to cover its cash needs for the next 12 months and "thereafter for the foreseeable future." This covers investments in infrastructure and AI initiatives and any return of capital to stockholders.
- Meta may also seek additional capital through debt, equity

{'analysis': {'query_type': 'qualitative',
  'tickers': ['META'],
  'sql_keywords': ['Purchase of property and equipment',
   'Property and equipment, net'],
  'period_granularity': 'any',
  'date_start': '',
  'date_end': '',
  'narrative_search_query': 'Meta AI infrastructure investments capital expenditures data centers servers GPUs outlook strategy'},
 'narrative_data':   ticker filing_type filing_date                           section  \
 0   META        10-Q  2026-10-22  Risk Factors - Product Offerings   
 1   META        10-K  2026-01-29                          Overview   
 2   META        10-K  2025-01-30                          Overview   
 3   META        10-Q  2026-04-30        Material Cash Requirements   
 4   META        10-Q  2026-07-30        Material Cash Requirements   
 
                                                                         chunk_text  \
 0  We have made significant investments in AI initiatives, including generative...   
 1  We build technolog

### Simple Query Interface Examples from README

In [9]:
ask("How much did Amazon spend on infrastructure in 2025?")


QUERY: How much did Amazon spend on infrastructure in 2025?



Query type: quantitative
Tickers: ['AMZN']
SQL keywords: ['Purchase of property and equipment', 'Property and equipment, net', 'Technology and infrastructure', 'Capital expenditures']
Granularity: annual
Date range: 2025-01-01 to 2025-12-31

🔢 Searching financial facts...
   Found 3 financial facts

🤖 Synthesizing answer with Claude...



ANSWER
Amazon's filings in the data have no single line called "infrastructure spending." Two reported figures could fit, depending on what you mean:

| Measure (FY2025, Jan 1 to Dec 31, 2025) | Amount | Source |
|---|---|---|
| Purchases of property and equipment (capital spending) | **$131,819M** | 10-K filed 2026-02-06 |
| "Technology and infrastructure" (reported expense line) | **$108,521M** | 10-K filed 2026-02-06 |

- **Capital investment in infrastructure:** The best fit is purchases of property and equipment, at about $131.8B. Net property and equipment was $357,025M at 2025-12-31.
- **Operating expense:** "Technology and infrastructure" was about $108.5B. It is a reported expense line, not capital spending. It should not be added to the purchases figure, because the data does not say whether the two overlap.

The data does not break out how much of the $131.8B went to specific infrastructure such as data centers, servers, or fulfillment. It also does not say what the "Techno

{'analysis': {'query_type': 'quantitative',
  'tickers': ['AMZN'],
  'sql_keywords': ['Purchase of property and equipment',
   'Property and equipment, net',
   'Technology and infrastructure',
   'Capital expenditures'],
  'period_granularity': 'annual',
  'date_start': '2025-01-01',
  'date_end': '2025-12-31',
  'narrative_search_query': ''},
 'financial_data':   ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0   AMZN         2025             4.0       instant         None  2025-12-31   
 1   AMZN         2025             NaN            FY   2025-01-01  2025-12-31   
 2   AMZN         2025             NaN            FY   2025-01-01  2025-12-31   
 
      statement_type                                label     value  \
 0     balance_sheet          Property and equipment, net  357025.0   
 1          cashflow  Purchases of property and equipment  131819.0   
 2  income_statement        Technology and infrastructure  108521.0   
 
            unit  is_de

In [10]:
ask("What risks does Microsoft mention about AI and power?")


QUERY: What risks does Microsoft mention about AI and power?



Query type: qualitative
Tickers: ['MSFT']
SQL keywords: []
Granularity: any
Narrative query: Microsoft risk factors AI data center power electricity availability constraints energy supply grid capacity infrastructure

📄 Searching narrative chunks...
   Found 5 relevant chunks

🤖 Synthesizing answer with Claude...



ANSWER
**Short answer:** Microsoft's filings tie AI growth to rising computing demand, and they name insufficient or unavailable power as one of several operational failures that could hurt the business. The text does not quantify any of this.

**What the filings say** (Operational Risks section; the 10-K filed 2023-07-27 and the 10-Qs filed 2023-10-24, 2024-01-30 and 2024-04-25)

- **AI drives computing demand.** User traffic, service growth and product complexity "demand more computing power." Demand keeps rising as Microsoft adds AI features and functionality to Bing, Azure, Microsoft 365, Teams, Dynamics 365, OneDrive, SharePoint Online, Skype, Xbox, Outlook.com and Microsoft Account services.
- **Datacenters depend on energy.** They rely on "permitted and buildable land, predictable energy, networking supplies, and servers, including GPUs and other components."
- **Cost and availability of those dependencies could be hit by:**
  - the transition to a clean energy economy
  - loca

{'analysis': {'query_type': 'qualitative',
  'tickers': ['MSFT'],
  'sql_keywords': [],
  'period_granularity': 'any',
  'date_start': '',
  'date_end': '',
  'narrative_search_query': 'Microsoft risk factors AI data center power electricity availability constraints energy supply grid capacity infrastructure'},
 'narrative_data':   ticker filing_type filing_date                           section  \
 0   MSFT        10-Q  2024-04-25  Risk Factors - Operational Risks   
 1   MSFT        10-Q  2024-01-30  Risk Factors - Operational Risks   
 2   MSFT        10-Q  2023-10-24  Risk Factors - Operational Risks   
 3   MSFT        10-Q  2024-04-25  Risk Factors - Operational Risks   
 4   MSFT        10-K  2023-07-27  Risk Factors - Operational Risks   
 
                                                                         chunk_text  \
 0  ### OPERATIONAL RISKS\nWe may have excessive outages, data losses, and disru...   
 1  ### OPERATIONAL RISKS\nWe may have excessive outages, data loss

In [11]:
ask("Compare data center spending across the hyperscalers")


QUERY: Compare data center spending across the hyperscalers



Query type: both
Tickers: ['AMZN', 'GOOGL', 'META', 'MSFT', 'ORCL']
SQL keywords: ['Purchase of property and equipment', 'Additions to property and equipment', 'Capital expenditures', 'Property and equipment, net']
Granularity: any
Narrative query: data center capital expenditures investment AI infrastructure spending plans hyperscalers

🔢 Searching financial facts...


   Found 100 financial facts

📄 Searching narrative chunks...
   Found 10 relevant chunks

🤖 Synthesizing answer with Claude...



ANSWER
# Data center spending across the hyperscalers

None of the filings in the data break out data center spending on its own. The closest proxy is capital expenditure (purchases of, or additions to, property and equipment), which also includes offices, servers, networking and other assets.

## Latest quarter (roughly Jul–Sep 2026)

| Company | Metric | Period | Amount |
|---|---|---|---|
| AMZN | Purchases of P&E | FY2026 Q3 (Jul 1–Sep 30, 2026) | $77,259M |
| GOOGL | Purchases of P&E (derived) | FY2026 Q3 (Jul 1–Sep 30, 2026) | $51,355M |
| MSFT | Additions to P&E | FY2027 Q1 (Jul 1–Sep 30, 2026) | $39,367M |
| META | Purchases of P&E (derived) | FY2026 Q3 (Jul 1–Sep 30, 2026) | $33,942M |
| ORCL | Capital expenditures | FY2027 Q1 (Jun 1–Aug 31, 2026) | $28,499M |

- Combined, the five spent about $230B in the latest quarter. Amazon was the largest at about a third of that.
- Amazon's FY2026 Q3 spend was about 2.2x its year-earlier quarter ($35,095M in FY2025 Q3).
- Amazon's quar

{'analysis': {'query_type': 'both',
  'tickers': ['AMZN', 'GOOGL', 'META', 'MSFT', 'ORCL'],
  'sql_keywords': ['Purchase of property and equipment',
   'Additions to property and equipment',
   'Capital expenditures',
   'Property and equipment, net'],
  'period_granularity': 'any',
  'date_start': '',
  'date_end': '',
  'narrative_search_query': 'data center capital expenditures investment AI infrastructure spending plans hyperscalers'},
 'financial_data':    ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0    AMZN         2026             3.0       instant         None  2026-09-30   
 1    AMZN         2026             2.0       instant         None  2026-06-30   
 2    AMZN         2026             1.0       instant         None  2026-03-31   
 3    AMZN         2025             4.0       instant         None  2025-12-31   
 4    AMZN         2025             3.0       instant         None  2025-09-30   
 ..    ...          ...             ...       

### Quantitative Analysis examples from README

In [12]:
ask("What was Microsoft's total revenue in Q2 2025?")


QUERY: What was Microsoft's total revenue in Q2 2025?



Query type: quantitative
Tickers: ['MSFT']
SQL keywords: ['Total revenue', 'Revenue', 'Net sales']
Granularity: quarterly
Date range: 2024-10-01 to 2025-06-30

🔢 Searching financial facts...
   Found 15 financial facts

🤖 Synthesizing answer with Claude...



ANSWER
**Microsoft's revenue for fiscal Q2 2025 (Oct 1 – Dec 31, 2024) was $69,632M.** This is the latest reported figure, taken from the 10-Q filed 2026-01-28, where the period appears as a comparative.

Microsoft's fiscal year ends in June, so "Q2 2025" could also mean calendar Q2 2025 (Apr 1 – Jun 30, 2025). That period is Microsoft's FY2025 Q4, with revenue of **$76,441M**. This figure is derived as full-year revenue minus the nine-month year-to-date total, and it comes from the 10-K filed 2026-07-29.

| Period | Dates | Revenue |
|---|---|---|
| FY2025 Q2 (fiscal) | 2024-10-01 to 2024-12-31 | $69,632M |
| FY2025 Q4 (calendar Q2 2025) | 2025-04-01 to 2025-06-30 | $76,441M (derived) |



{'analysis': {'query_type': 'quantitative',
  'tickers': ['MSFT'],
  'sql_keywords': ['Total revenue', 'Revenue', 'Net sales'],
  'period_granularity': 'quarterly',
  'date_start': '2024-10-01',
  'date_end': '2025-06-30',
  'narrative_search_query': ''},
 'financial_data':    ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0    MSFT         2025               4             Q   2025-04-01  2025-06-30   
 1    MSFT         2025               3             Q   2025-01-01  2025-03-31   
 2    MSFT         2025               2             Q   2024-10-01  2024-12-31   
 3    MSFT         2025               4       instant         None  2025-06-30   
 4    MSFT         2025               3       instant         None  2025-03-31   
 5    MSFT         2025               2       instant         None  2024-12-31   
 6    MSFT         2025               4             Q   2025-04-01  2025-06-30   
 7    MSFT         2025               3             Q   2025-01-01  20

In [13]:
ask("Compare operating expenses between Meta and Alphabet in 2024")


QUERY: Compare operating expenses between Meta and Alphabet in 2024



Query type: quantitative
Tickers: ['META', 'GOOGL']
SQL keywords: ['Total operating expenses', 'Total costs and expenses', 'Research and development', 'General and administrative', 'Marketing and sales']
Granularity: annual
Date range: 2024-01-01 to 2024-12-31

🔢 Searching financial facts...
   Found 7 financial facts

🤖 Synthesizing answer with Claude...



ANSWER
# Operating Expenses: Meta vs. Alphabet, FY2024

Both companies use a calendar fiscal year (Jan 1 – Dec 31, 2024), so the periods align directly.

| Line item (FY2024) | Meta | Alphabet (GOOGL) |
|---|---|---|
| Research and development | $43,873M | $49,326M |
| General and administrative | $9,740M | $14,188M |
| Marketing and sales | $11,347M | Not in data |
| **Total costs and expenses** | **$95,121M** | **$237,628M** |

Sources: Meta 10-K filed 2026-01-29; Alphabet 10-K filed 2026-02-05. Both are the latest reported figures for FY2024.

**Key takeaways**
- **Total costs and expenses:** Alphabet's $237,628M is about 2.5x Meta's $95,121M, a difference of $142,507M.
- **R&D:** Alphabet spent $5,453M more than Meta ($49,326M vs. $43,873M). R&D is about 46% of Meta's total costs and expenses (43,873 / 95,121) and about 21% of Alphabet's (49,326 / 237,628).
- **G&A:** Alphabet's $14,188M exceeds Meta's $9,740M by $4,448M.

**Caveats and missing data**
- "Total costs and expenses" 

{'analysis': {'query_type': 'quantitative',
  'tickers': ['META', 'GOOGL'],
  'sql_keywords': ['Total operating expenses',
   'Total costs and expenses',
   'Research and development',
   'General and administrative',
   'Marketing and sales'],
  'period_granularity': 'annual',
  'date_start': '2024-01-01',
  'date_end': '2024-12-31',
  'narrative_search_query': ''},
 'financial_data':   ticker  fiscal_year fiscal_quarter fiscal_period period_start  period_end  \
 0  GOOGL         2024           None            FY   2024-01-01  2024-12-31   
 1  GOOGL         2024           None            FY   2024-01-01  2024-12-31   
 2  GOOGL         2024           None            FY   2024-01-01  2024-12-31   
 3   META         2024           None            FY   2024-01-01  2024-12-31   
 4   META         2024           None            FY   2024-01-01  2024-12-31   
 5   META         2024           None            FY   2024-01-01  2024-12-31   
 6   META         2024           None            FY 

In [14]:
ask("Show me Amazon's cash flow trends over the past 8 quarters")


QUERY: Show me Amazon's cash flow trends over the past 8 quarters



Query type: quantitative
Tickers: ['AMZN']
SQL keywords: ['Net cash provided by operating activities', 'Net cash used in investing activities', 'Net cash provided by (used in) financing activities', 'Purchase of property and equipment', 'Cash and cash equivalents']
Granularity: quarterly
Date range: 2024-10-01 to 2026-11-08

🔢 Searching financial facts...
   Found 60 financial facts

🤖 Synthesizing answer with Claude...



ANSWER
# Amazon (AMZN) cash flow, past 8 quarters (FY2024 Q4 to FY2026 Q3)

Amazon's fiscal year is the calendar year. All figures are in $M. Operating, investing and capex data for FY2024 Q4 are missing from the retrieved data, so the full cash flow picture covers 7 quarters.

| Quarter | Operating CF | Capex (purchases of P&E) | Investing CF | Financing CF | FX effect | Net change in cash | Ending cash (incl. restricted) |
|---|---|---|---|---|---|---|---|
| FY24 Q4 (Dec-24) | n/a | n/a | n/a | -3,308* | -1,250* | n/a | 82,312 |
| FY25 Q1 (Mar-25) | 17,015 | 25,019 | -29,803 | -47 | 416 | -12,419 | 69,893 |
| FY25 Q2 (Jun-25) | 32,515 | 32,183 | -39,424 | -2,539 | 1,008 | -8,440 | 61,453 |
| FY25 Q3 (Sep-25) | 35,525 | 35,095 | -26,073 | -44 | -397 | 9,011 | 70,464 |
| FY25 Q4 (Dec-25) | 54,459* | 39,522* | -47,245* | 12,291* | 137* | 19,642* | 90,106 |
| FY26 Q1 (Mar-26) | 26,032 | 44,203 | -64,212 | 52,767 | -1 | 14,586 | 104,692 |
| FY26 Q2 (Jun-26) | 45,387 | 54,208 | -79,245 | 

{'analysis': {'query_type': 'quantitative',
  'tickers': ['AMZN'],
  'sql_keywords': ['Net cash provided by operating activities',
   'Net cash used in investing activities',
   'Net cash provided by (used in) financing activities',
   'Purchase of property and equipment',
   'Cash and cash equivalents'],
  'period_granularity': 'quarterly',
  'date_start': '2024-10-01',
  'date_end': '2026-11-08',
  'narrative_search_query': ''},
 'financial_data':    ticker  fiscal_year  fiscal_quarter fiscal_period period_start  period_end  \
 0    AMZN         2026               3       instant         None  2026-09-30   
 1    AMZN         2026               2       instant         None  2026-06-30   
 2    AMZN         2026               1       instant         None  2026-03-31   
 3    AMZN         2025               4       instant         None  2025-12-31   
 4    AMZN         2025               3       instant         None  2025-09-30   
 5    AMZN         2025               2       instant  